In [70]:
import numpy as np
import os
import joblib
import pandas as pd
import pickle
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import vit_b_16
import torch.nn.functional as F

#### Loading datasets

In [29]:
data = joblib.load("datasets.pkl")
X_train_flat = data["X_train_flat"]
X_train_rad  = data["X_train_rad"]
y_train      = data["y_train"]
X_valid_flat = data["X_valid_flat"]
X_valid_rad  = data["X_valid_rad"]
y_valid      = data["y_valid"]

In [30]:
X_train_img = X_train_flat.reshape(-1,1,224,224)
X_valid_img = X_valid_flat.reshape(-1,1,224,224)

In [31]:
np.save("X_train_img.npy", X_train_img)
np.save("X_valid_img.npy", X_valid_img)

#### Random Forest 2

In [74]:
scaler_rad = StandardScaler()
X_train_rad = scaler_rad.fit_transform(X_train_rad)
X_valid_rad  = scaler_rad.transform(X_valid_rad)

scaler_flat = StandardScaler()
X_train_flat = scaler_flat.fit_transform(X_train_flat)
X_valid_flat  = scaler_flat.transform(X_valid_flat)

pca = PCA(n_components=50, random_state=42)
X_train_pca = pca.fit_transform(X_train_flat)
X_valid_pca  = pca.transform(X_valid_flat)

X_train_mix = np.hstack([X_train_pca, X_train_rad])
X_valid_mix  = np.hstack([X_valid_pca,  X_valid_rad])

In [89]:
scaler_rad = StandardScaler()
X_train_rad = scaler_rad.fit_transform(X_train_rad)
X_valid_rad  = scaler_rad.transform(X_valid_rad)

scaler_flat = StandardScaler()
X_train_flat = scaler_flat.fit_transform(X_train_flat)
X_valid_flat  = scaler_flat.transform(X_valid_flat)

pca = PCA(n_components=135, random_state=42)
X_train_pca = pca.fit_transform(X_train_flat)
X_valid_pca  = pca.transform(X_valid_flat)

X_train_mix = np.hstack([X_train_pca, X_train_rad])
X_valid_mix  = np.hstack([X_valid_pca,  X_valid_rad])

In [90]:
with open("preprocess3.pkl", "wb") as f:
    pickle.dump({"scaler": scaler_rad, "scaler_flat": scaler_flat, "pca": pca}, f)

Wykorzystano podstawowe hiperparametry, głębokość mniejszą aby uniknąć overfittingu i manualne dobranie wag, aby wykazać większą wagę dla obrazów z guzami złośliwymi

In [75]:
rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=10,
    min_samples_leaf=2,
    class_weight={0:1, 1:2}, 
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_mix, y_train)

,n_estimators,800
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [76]:
proba = rf.predict_proba(X_valid_mix)[:,1]

y_pred = (proba >= 0.3).astype(int)

print("ROC-AUC:", roc_auc_score(y_valid, proba))
print("Recall :", recall_score(y_valid, y_pred))
print("Precision:", precision_score(y_valid, y_pred))
print("Pred dist:", np.bincount(y_pred))

ROC-AUC: 0.5969654657016992
Recall : 0.9911894273127754
Precision: 0.34142640364188165
Pred dist: [ 16 659]


In [86]:
joblib.dump(rf_pca, "rf_pca2.pkl")
joblib.dump(rf_rad, "rf_rad2.pkl")
joblib.dump(rf, "rf_mix2.pkl")

['rf_mix2.pkl']

In [88]:
print(rf_pca.n_features_in_)

135


#### CNN 

Zdecydowano na płytką, ale stabilną architekturę, ponieważ zbiór danych jest stosunkowo niewielki, obrazy mają subtelne różnice, zbyt głęboka sieć mogłaby doprowadzić do przeuczenia, funkcja ReLu jest standardem w CNN dla obrazów medycznych i zachowuje prostotę i stabilność modelu, pooling redukucje rozmiar cech a dropout jest formą regularyzacji więc redukuje przeuczenie. Funckja CrossEntropyLoss umożliwia klasyfikację binarną i pozwala na użycie wag

In [78]:
X_train_img = torch.tensor(X_train_flat.reshape(-1,1,224,224), dtype=torch.float32)
y_train_t   = torch.tensor(y_train, dtype=torch.long)

X_valid_img  = torch.tensor(X_valid_flat.reshape(-1,1,224,224), dtype=torch.float32)
y_valid_t    = torch.tensor(y_valid, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train_img, y_train_t),
    batch_size=16,
    shuffle=True)

In [79]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(32*(224//4)*(224//4), 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,2)
        )

    def forward(self,x):
        return self.net(x)

In [80]:
device = "cuda" if torch.cuda.is_available() else "cpu"

cnn = CNN().to(device)

weights = torch.tensor(
    [1.0, 1.5], device=device
)

loss_fn = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)

In [81]:
epochs = 10
for epoch in range(epochs):
    cnn.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        opt.zero_grad()
        out = cnn(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10 | Loss: 0.7977
Epoch 2/10 | Loss: 0.6777
Epoch 3/10 | Loss: 0.6237
Epoch 4/10 | Loss: 0.5259
Epoch 5/10 | Loss: 0.3929
Epoch 6/10 | Loss: 0.2516
Epoch 7/10 | Loss: 0.1827
Epoch 8/10 | Loss: 0.1294
Epoch 9/10 | Loss: 0.1168
Epoch 10/10 | Loss: 0.0960


In [84]:
cnn.eval()
with torch.no_grad():
    logits = cnn(X_valid_img.to(device))
    proba  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

y_pred = (proba >= 0.4).astype(int)
print("ROC-AUC:", roc_auc_score(y_valid, proba))
print("Recall :", recall_score(y_valid, y_pred))
print("Precision:", precision_score(y_valid, y_pred))

ROC-AUC: 0.5805931403398363
Recall : 0.4008810572687225
Precision: 0.40625


In [85]:
torch.save(cnn.state_dict(), "cnn_model2.pt")

#### Transformer ViT

Wybrano model ViT-B/16 poniważ patch size pozwala na utrzymanie kompromisu między wykrywaniem szczegółów a kosztem obliczeń, większość wag encoderów została zamrożona i trenowanie na kilku warstwach aby skrócić czas treningu, analogicznie jak dla CNN funckja CrossEntropyLoss pozwala na klasyfikacje binarną i zwiększa czułość modelu na klasę mniejszościową

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

C:\Users\Dell\anaconda3\envs\breast_cancer\lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11000). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\cb\pytorch_1000000000000\work\c10\cuda\CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [16]:
X = X_train_flat.reshape(-1, 224, 224)
y = y_train

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [17]:
X = X.unsqueeze(1).repeat(1, 3, 1, 1)
#normalizacja ImageNet
mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)
X = (X - mean) / std

In [18]:
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vit = vit_b_16(weights="IMAGENET1K_V1")
vit.heads.head = nn.Linear(vit.heads.head.in_features, 2)
vit = vit.to(device)

In [20]:
weights = compute_class_weight("balanced", classes=np.array([0,1]), y=y.numpy())
weights = torch.tensor(weights, dtype=torch.float32).to(device)

loss_fn = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(vit.parameters(), lr=3e-5)

In [21]:
epochs = 8
for epoch in range(epochs):
    vit.train()
    total_loss = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        opt.zero_grad()
        preds = vit(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")

Epoch 1/8 | Loss: 0.6824
Epoch 2/8 | Loss: 0.6350
Epoch 3/8 | Loss: 0.5227
Epoch 4/8 | Loss: 0.3110
Epoch 5/8 | Loss: 0.1643
Epoch 6/8 | Loss: 0.0985
Epoch 7/8 | Loss: 0.0860
Epoch 8/8 | Loss: 0.0376


In [22]:
torch.save(vit.state_dict(), "vit_model.pt")

In [23]:
xb, yb = next(iter(loader))
print(xb.shape)   
print(xb.min(), xb.max(), xb.mean())

torch.Size([16, 3, 224, 224])
tensor(-2.1179) tensor(2.6400) tensor(-1.1875)


In [ ]:
print("Total training time:", (time.time()-start)/60, "minutes")

Total training time: 284.22311841249467 minutes


In [ ]:
torch.save(vit.state_dict(), "vit_model.pt")